# 원룸 / 오피스텔 RandomForest 회귀 모델 분리 학습

이 노트북은 아래 두 CSV를 각각 따로 학습합니다.

- `oneroom_processed_items_data_2026-04-22.csv`
- `officetel_processed_items_data_2026-04-22.csv`

목표값은 `converted_monthly_rent`이고, 각 데이터셋별로 상위 1% 이상치를 제거한 뒤 `RandomForestRegressor`를 학습합니다.

학습된 모델은 백엔드에서 불러 쓸 수 있도록 `.joblib` 파일로 저장합니다.

## 1. 라이브러리 import

In [31]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
sns.set_theme(style="whitegrid")

## 2. 경로 설정

주피터 노트북 실행 위치가 프로젝트 루트가 아니어도 돌아가도록, 기본 프로젝트 경로를 한 번 더 잡아줍니다.

In [32]:
BASE_DIR = Path(".")
MODEL_DIR = BASE_DIR / "models"
FIGURE_DIR = MODEL_DIR / "figures"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

ONEROOM_CSV = BASE_DIR / "oneroom_processed_items_data_2026-04-22.csv"
OFFICETEL_CSV = BASE_DIR / "officetel_processed_items_data_2026-04-22.csv"

print("BASE_DIR:", BASE_DIR.resolve())
print("ONEROOM exists:", ONEROOM_CSV.exists())
print("OFFICETEL exists:", OFFICETEL_CSV.exists())

BASE_DIR: C:\Users\Playdata\Desktop\SKN_Final_1team\ml_research\RandomForest
ONEROOM exists: True
OFFICETEL exists: True


## 3. 공통 설정

In [33]:
TARGET_COLUMN = "converted_monthly_rent"
RANDOM_STATE = 42
TEST_SIZE = 0.2
OUTLIER_QUANTILE = 0.995

## 4. 데이터 확인

In [34]:
oneroom_df = pd.read_csv(ONEROOM_CSV)
officetel_df = pd.read_csv(OFFICETEL_CSV)

print("oneroom:", oneroom_df.shape)
print("officetel:", officetel_df.shape)

display(oneroom_df.head())
display(officetel_df.head())

oneroom: (25548, 86)
officetel: (20776, 86)


,manage_cost,floor,all_floors,area_m2,has_parking,has_elevator,bathroom_count,movein_date,has_air_con,has_fridge,...,district_양천구,district_영등포구,district_용산구,district_은평구,district_종로구,district_중구,district_중랑구,contract_class_반전세,contract_class_월세,contract_class_전세
0,5,3,5,17.98,False,False,1,1,True,True,...,False,False,False,False,False,False,False,False,True,False
1,0,5,5,20.00,False,False,1,1,True,True,...,False,False,False,False,False,False,False,False,True,False
2,2,2,3,24.00,False,False,1,1,True,True,...,False,False,False,False,False,False,False,False,True,False
3,7,2,5,19.83,True,True,1,1,True,True,...,False,False,False,False,False,False,False,False,True,False
4,8,2,3,20.00,False,False,1,1,True,True,...,False,False,False,False,False,False,False,False,False,True


,manage_cost,floor,all_floors,area_m2,has_parking,has_elevator,bathroom_count,movein_date,has_air_con,has_fridge,...,district_양천구,district_영등포구,district_용산구,district_은평구,district_종로구,district_중구,district_중랑구,contract_class_반전세,contract_class_월세,contract_class_전세
0,12,10,12,19.24,True,True,1,55,True,True,...,False,False,False,False,False,False,False,False,True,False
1,15,13,16,25.77,True,True,1,92,True,True,...,False,False,False,False,False,False,False,False,True,False
2,0,9,15,24.00,True,True,1,33,True,True,...,False,False,False,False,False,False,True,False,False,True
3,12,5,6,25.67,True,True,1,60,True,True,...,False,False,False,False,False,False,False,False,False,True
4,8,6,10,21.96,True,True,1,14,True,True,...,False,False,False,False,False,False,False,False,True,False


In [ ]:
display(oneroom_df[TARGET_COLUMN].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.995]))
display(officetel_df[TARGET_COLUMN].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.995]))

count     25548.000000
mean         71.190279
std        2064.688207
min           8.083333
1%           20.206833
5%           28.137500
25%          42.150000
50%          54.583333
75%          70.315000
95%         100.675000
99%         130.833333
max      330051.000000
Name: converted_monthly_rent, dtype: float64

count     20776.000000
mean        148.177136
std        5226.049894
min           4.250000
1%           36.375000
5%           47.291667
25%          67.916667
50%          92.000000
75%         126.666667
95%         214.750000
99%         372.500000
max      750004.583333
Name: converted_monthly_rent, dtype: float64

## 5. 학습 함수 정의

In [36]:
def remove_target_outliers(df, target_column=TARGET_COLUMN, quantile=OUTLIER_QUANTILE):
    target = pd.to_numeric(df[target_column], errors="coerce")
    upper_limit = target.quantile(quantile)
    clean_df = df[target <= upper_limit].copy()
    removed_count = len(df) - len(clean_df)
    return clean_df, upper_limit, removed_count


def make_xy(df, target_column=TARGET_COLUMN):
    X = df.drop(columns=[target_column]).copy()
    y = pd.to_numeric(df[target_column], errors="coerce")

    bool_columns = X.select_dtypes(include=["bool"]).columns
    X[bool_columns] = X[bool_columns].astype(int)

    X = X.apply(pd.to_numeric, errors="coerce").fillna(0)

    valid_index = y.notna()
    X = X.loc[valid_index]
    y = y.loc[valid_index]

    return X, y


def train_random_forest(X_train, y_train):
    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=RANDOM_STATE,
        n_jobs=1,
    )
    model.fit(X_train, y_train)
    return model


def evaluate_regression(y_true, y_pred):
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "r2": r2_score(y_true, y_pred),
    }


def train_one_dataset(dataset_name, df, source_csv):
    clean_df, upper_limit, removed_count = remove_target_outliers(df)
    X, y = make_xy(clean_df)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )

    model = train_random_forest(X_train, y_train)
    pred = model.predict(X_test)
    metrics = evaluate_regression(y_test, pred)

    artifact = {
        "model": model,
        "features": X.columns.tolist(),
        "target_column": TARGET_COLUMN,
        "dataset_name": dataset_name,
        "source_csv": str(source_csv),
        "outlier_rule": f"{TARGET_COLUMN} <= {OUTLIER_QUANTILE} quantile",
        "outlier_upper_limit": float(upper_limit),
        "removed_outlier_count": int(removed_count),
        "train_rows": int(len(X_train)),
        "test_rows": int(len(X_test)),
        "metrics": {key: float(value) for key, value in metrics.items()},
    }

    model_path = MODEL_DIR / f"{dataset_name}_random_forest_model.joblib"
    joblib.dump(artifact, model_path)

    result_df = pd.DataFrame({
        "actual": y_test,
        "pred": pred,
    })
    result_df["error"] = result_df["actual"] - result_df["pred"]
    result_df["abs_error"] = result_df["error"].abs()

    summary = {
        "dataset_name": dataset_name,
        "source_csv": str(source_csv),
        "model_path": str(model_path),
        "rows_before": len(df),
        "rows_after": len(X),
        "removed_outlier_count": removed_count,
        "outlier_upper_limit": upper_limit,
        "mae": metrics["mae"],
        "rmse": metrics["rmse"],
        "r2": metrics["r2"],
    }

    return {
        "dataset_name": dataset_name,
        "clean_df": clean_df,
        "X": X,
        "y": y,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "pred": pred,
        "model": model,
        "artifact": artifact,
        "model_path": model_path,
        "result_df": result_df,
        "summary": summary,
    }

## 6. 원룸 모델 학습

In [37]:
oneroom_result = train_one_dataset(
    dataset_name="oneroom",
    df=oneroom_df,
    source_csv=ONEROOM_CSV,
)

oneroom_result["summary"]

KeyboardInterrupt: 

## 7. 오피스텔 모델 학습

In [ ]:
officetel_result = train_one_dataset(
    dataset_name="officetel",
    df=officetel_df,
    source_csv=OFFICETEL_CSV,
)

officetel_result["summary"]

## 8. 성능 비교표 저장

In [ ]:
summary_df = pd.DataFrame([
    oneroom_result["summary"],
    officetel_result["summary"],
])

summary_path = MODEL_DIR / "training_results.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

display(summary_df)
print("saved:", summary_path)

## 9. 성능 비교 시각화

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.barplot(data=summary_df, x="dataset_name", y="mae", ax=axes[0])
axes[0].set_title("MAE lower is better")

sns.barplot(data=summary_df, x="dataset_name", y="rmse", ax=axes[1])
axes[1].set_title("RMSE lower is better")

sns.barplot(data=summary_df, x="dataset_name", y="r2", ax=axes[2])
axes[2].set_title("R2 higher is better")
axes[2].set_ylim(0, 1)

plt.tight_layout()
compare_fig_path = FIGURE_DIR / "model_metric_comparison.png"
plt.savefig(compare_fig_path, dpi=150, bbox_inches="tight")
plt.show()

print("saved:", compare_fig_path)

## 10. 실제값 vs 예측값 시각화

In [ ]:
def plot_actual_vs_pred(result, save=True):
    dataset_name = result["dataset_name"]
    y_test = result["y_test"]
    pred = result["pred"]

    plt.figure(figsize=(6, 6))
    sns.scatterplot(x=y_test, y=pred, alpha=0.35)

    min_value = min(y_test.min(), pred.min())
    max_value = max(y_test.max(), pred.max())
    plt.plot([min_value, max_value], [min_value, max_value], color="red", linewidth=2)

    plt.title(f"{dataset_name} actual vs predicted")
    plt.xlabel("Actual converted monthly rent")
    plt.ylabel("Predicted converted monthly rent")
    plt.tight_layout()

    if save:
        path = FIGURE_DIR / f"{dataset_name}_actual_vs_pred.png"
        plt.savefig(path, dpi=150, bbox_inches="tight")
        print("saved:", path)

    plt.show()


plot_actual_vs_pred(oneroom_result)
plot_actual_vs_pred(officetel_result)

## 11. 오차 분포 시각화

In [ ]:
def plot_error_distribution(result, save=True):
    dataset_name = result["dataset_name"]
    result_df = result["result_df"]

    plt.figure(figsize=(8, 4))
    sns.histplot(result_df["error"], bins=50, kde=True)
    plt.axvline(0, color="red", linewidth=2)
    plt.title(f"{dataset_name} prediction error distribution")
    plt.xlabel("actual - predicted")
    plt.tight_layout()

    if save:
        path = FIGURE_DIR / f"{dataset_name}_error_distribution.png"
        plt.savefig(path, dpi=150, bbox_inches="tight")
        print("saved:", path)

    plt.show()


plot_error_distribution(oneroom_result)
plot_error_distribution(officetel_result)

## 12. 중요 변수 시각화

In [ ]:
def get_feature_importance(result):
    return pd.DataFrame({
        "feature": result["X"].columns,
        "importance": result["model"].feature_importances_,
    }).sort_values("importance", ascending=False)


def plot_feature_importance(result, top_n=20, save=True):
    dataset_name = result["dataset_name"]
    importance_df = get_feature_importance(result).head(top_n)

    plt.figure(figsize=(10, 7))
    sns.barplot(data=importance_df, x="importance", y="feature")
    plt.title(f"{dataset_name} top {top_n} feature importances")
    plt.tight_layout()

    if save:
        path = FIGURE_DIR / f"{dataset_name}_feature_importance.png"
        plt.savefig(path, dpi=150, bbox_inches="tight")
        print("saved:", path)

    plt.show()
    return importance_df


oneroom_importance = plot_feature_importance(oneroom_result)
officetel_importance = plot_feature_importance(officetel_result)

## 13. 예측 결과 중 오차가 큰 행 확인

In [ ]:
display(oneroom_result["result_df"].sort_values("abs_error", ascending=False).head(20))
display(officetel_result["result_df"].sort_values("abs_error", ascending=False).head(20))

## 14. 저장된 모델 불러오기 테스트

백엔드에서는 `joblib.load()`로 artifact를 불러온 뒤, `artifact["model"]`과 `artifact["features"]`를 사용하면 됩니다.

In [ ]:
loaded_oneroom = joblib.load(MODEL_DIR / "oneroom_random_forest_model.joblib")
loaded_officetel = joblib.load(MODEL_DIR / "officetel_random_forest_model.joblib")

print(loaded_oneroom.keys())
print(loaded_officetel.keys())
print(loaded_oneroom["metrics"])
print(loaded_officetel["metrics"])

## 15. 백엔드 예측 함수 예시

In [ ]:
def preprocess_for_saved_model(input_data, feature_columns):
    if isinstance(input_data, dict):
        X = pd.DataFrame([input_data])
    else:
        X = input_data.copy()

    bool_columns = X.select_dtypes(include=["bool"]).columns
    X[bool_columns] = X[bool_columns].astype(int)

    X = X.apply(pd.to_numeric, errors="coerce").fillna(0)
    X = X.reindex(columns=feature_columns, fill_value=0)

    return X


def predict_with_artifact(artifact, input_data):
    model = artifact["model"]
    feature_columns = artifact["features"]
    X = preprocess_for_saved_model(input_data, feature_columns)
    pred = model.predict(X)
    return float(pred[0]) if isinstance(input_data, dict) else pred

In [ ]:
# 예시: 테스트 데이터 첫 행으로 예측
sample_oneroom = oneroom_result["X_test"].iloc[0].to_dict()
sample_officetel = officetel_result["X_test"].iloc[0].to_dict()

print("oneroom prediction:", predict_with_artifact(loaded_oneroom, sample_oneroom))
print("oneroom actual:", float(oneroom_result["y_test"].iloc[0]))

print("officetel prediction:", predict_with_artifact(loaded_officetel, sample_officetel))
print("officetel actual:", float(officetel_result["y_test"].iloc[0]))

## 16. 최종 산출물

노트북 실행 후 아래 파일들이 생성됩니다.

- `ml_research/RandomForest/models/oneroom_random_forest_model.joblib`
- `ml_research/RandomForest/models/officetel_random_forest_model.joblib`
- `ml_research/RandomForest/models/training_results.csv`
- `ml_research/RandomForest/models/figures/*.png`